# Compute M1 Preference-Aware Coefficients

This notebook runs Step 15 of the thesis prototype. It applies the first preference-to-coefficient mapping M1:

$$\delta_i \rightarrow R \rightarrow \lambda = f(p, R) \rightarrow \theta(\lambda)$$

The relationship matrix $R$ has already been computed from LoRA adapter geometry. M1 combines $R$ with example user preference vectors $p$ to produce corrected merge coefficients $\lambda$.

The notebook runs the existing coefficient-computation script and previews the resulting coefficient table.

## 1. Clone or update the repository

This cell always starts in `/content`. If `/content/master-thesis/.git` exists, it updates the repository. Otherwise, it clones a fresh copy. If a non-Git folder already occupies that path, the cell stops with a clear message instead of creating a nested repository.

In [ ]:
%cd /content

from pathlib import Path
import subprocess

repo_path = Path("/content/master-thesis")

if (repo_path / ".git").is_dir():
    print("Repository already exists. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
elif repo_path.exists():
    raise RuntimeError(
        "/content/master-thesis exists but is not a Git repository. "
        "Rename or remove that folder, then run this cell again."
    )
else:
    print("Cloning the repository...")
    subprocess.run(
        ["git", "clone", "https://github.com/NZhang137/master-thesis.git"],
        check=True,
    )

%cd /content/master-thesis

## 2. Inspect the repository

These commands confirm that Colab is in the repository root and show the available scripts and result files.

In [ ]:
!pwd
!ls
!ls scripts
!ls results

## 3. Install dependencies

M1 uses NumPy for its numerical calculations. Pandas is used only to display the CSV files as readable tables.

In [ ]:
!pip install -q pandas numpy

## 4. Check the relationship matrix

Step 15 requires `results/relationship_matrix.csv`. If the check below reports that the file is missing, run Step 14 first:

```bash
python scripts/compute_relationship_matrix.py
```

Step 14 requires the local helpful and harmless adapters.

In [ ]:
from pathlib import Path

matrix_path = Path("results/relationship_matrix.csv")

if matrix_path.is_file():
    print(f"Found input matrix: {matrix_path}")
else:
    print(f"Missing input matrix: {matrix_path}")
    print("Run Step 14 first: python scripts/compute_relationship_matrix.py")

## 5. Preview the relationship matrix

The CSV should contain a labeled $2 \times 2$ matrix with rows and columns for `helpful` and `harmless`.

In [ ]:
!cat results/relationship_matrix.csv

In [ ]:
import pandas as pd

relationship_df = pd.read_csv(
    "results/relationship_matrix.csv",
    index_col="adapter",
)
relationship_df

## 6. Compute the M1 coefficients

The script evaluates three example preferences and four correction strengths, applies the direct relationship-softmax mapping, and writes one small CSV file.

In [ ]:
!python scripts/compute_m1_coefficients.py

## 7. Inspect the M1 output

The output is saved as `results/m1_coefficients.csv`. It contains one row for each combination of preference vector and `tau` value.

In [ ]:
!cat results/m1_coefficients.csv

In [ ]:
m1_df = pd.read_csv("results/m1_coefficients.csv")
m1_df

## 8. Understand the output columns

- `method`: name of the coefficient mapping used for the row.
- `tau`: correction strength. At `tau = 0`, the output coefficients equal the normalized input preference.
- `p_helpful`: input preference weight for helpfulness.
- `p_harmless`: input preference weight for harmlessness.
- `lambda_helpful`: corrected merge coefficient for the helpful adapter.
- `lambda_harmless`: corrected merge coefficient for the harmless adapter.
- `score_helpful`: helpful component of the relationship score $R p$.
- `score_harmless`: harmless component of the relationship score $R p$.
- `l1_distance_to_p`: absolute coefficient change between $\lambda$ and $p$.
- `l2_distance_to_p`: Euclidean coefficient change between $\lambda$ and $p$.

For every row, the two lambda values should be non-negative and sum to one.

## 9. Git safety check

It is okay to commit the small `results/m1_coefficients.csv` result file.

Do **not** commit:

- `adapters/`
- `adapters.zip`
- `.safetensors` or `.bin` files
- checkpoints or full model files

In [ ]:
!git status

## What this notebook establishes

This notebook runs the first implemented $\lambda = f(p, R)$ mapping for a small set of example preferences and correction strengths, then records how far the corrected coefficients move from each preference vector inside the fixed interpolation family.